1 -  Top Customers by Sales (2016)

Functional Specification

- Join Sales.Orders (o) → Sales.OrderLines (ol) → Sales.Customers (c).
- Compute sales value as ol.UnitPrice * ol.Quantity.
- Filter orders to YEAR(o.OrderDate) = 2016.
- Group by customer; sort by total sales; take top 5.

In [ ]:
SELECT TOP 5
    c.CustomerName,
    SUM(ol.UnitPrice * ol.Quantity) AS TotalSales
FROM Sales.Orders AS o
JOIN Sales.OrderLines AS ol ON ol.OrderID = o.OrderID
JOIN Sales.Customers   AS c  ON c.CustomerID = o.CustomerID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY c.CustomerName
ORDER BY TotalSales DESC;


|CustomerName|TotalSales|
|------------|----------|
|Tailspin Toys (Arietta, NY)|91923.70|
|Tailspin Toys (Good Hart, MI)|91799.40|
|Wingtip Toys (Obetz, OH)|87221.90|
|Wingtip Toys (North Beach Haven, NJ)|85923.70|
|Wingtip Toys (Leathersville, GA)|79706.30|


---

2 - Best-Selling Stock Items (by Quantity, 2016)

Functional Specification

- Join Sales.Orders (o) → Sales.OrderLines (ol) → Warehouse.StockItems (si).
- Filter by order date year 2016.
- Sum quantities by item; take top 10.

In [ ]:
SELECT TOP 10
    si.StockItemName,
    SUM(ol.Quantity) AS TotalQty
FROM Sales.Orders AS o
JOIN Sales.OrderLines AS ol ON ol.OrderID = o.OrderID
JOIN Warehouse.StockItems AS si ON si.StockItemID = ol.StockItemID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY si.StockItemName
ORDER BY TotalQty DESC;


|StockItemName|TotalQty|
|-------------|--------|
|Black and orange fragile despatch tape 48mmx75m|26136|
|Black and orange fragile despatch tape 48mmx100m|25236|
|Clear packaging tape 48mmx75m|23400|
|Shipping carton (Brown) 356x356x279mm|21150|
|Chocolate beetles 250g|20424|
|Shipping carton (Brown) 413x285x187mm|20150|
|Shipping carton (Brown) 279x254x217mm|20075|
|Shipping carton (Brown) 356x229x229mm|19525|
|Shipping carton (Brown) 229x229x229mm|19475|
|Shipping carton (Brown) 457x457x457mm|19425|


---

3 - Monthly Sales Trend (2016)

Functional Specification

- Use Sales.Orders + Sales.OrderLines.
- Compute monthly buckets with YEAR/MONTH of OrderDate.
- Sum extended value.

In [ ]:
SELECT
    YEAR(o.OrderDate)  AS SalesYear,
    MONTH(o.OrderDate) AS SalesMonth,
    SUM(ol.UnitPrice * ol.Quantity) AS MonthlySales
FROM Sales.Orders AS o
JOIN Sales.OrderLines AS ol ON ol.OrderID = o.OrderID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY YEAR(o.OrderDate), MONTH(o.OrderDate)
ORDER BY SalesYear, SalesMonth;


|SalesYear|SalesMonth|MonthlySales|
|---------|----------|------------|
|2016|1|4612140.45|
|2016|2|4099480.35|
|2016|3|4807110.70|
|2016|4|4739058.60|
|2016|5|5138002.65|


---

4 - Customers with No Orders in 2016

Proposition: List all customers who did not place any orders in 2016 

Functional Specification:

- Start from Sales.Customers c.
- LEFT JOIN to Sales.Orders o filtered to 2016 on CustomerID.
- Keep rows where the joined order is NULL.

In [ ]:
SELECT
    c.CustomerName,
    c.PhoneNumber
FROM Sales.Customers AS c
LEFT JOIN Sales.Orders AS o
    ON o.CustomerID = c.CustomerID
   AND YEAR(o.OrderDate) = 2016
WHERE o.OrderID IS NULL
ORDER BY c.CustomerName;

NO DATA RETURNED

---

5 - Orders per Customer (count) in 2016

Proposition: Show how many orders each customer placed in 2016; list the top 10 by count.

Functional Specification:

- Use Sales.Orders o joined to Sales.Customers c.
- Filter to 2016; COUNT(*) per customer.
- Sort descending; take TOP 10.

In [ ]:
SELECT TOP 10
    c.CustomerName,
    COUNT(*) AS OrderCount2016
FROM Sales.Orders AS o
JOIN Sales.Customers AS c ON c.CustomerID = o.CustomerID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY c.CustomerName
ORDER BY OrderCount2016 DESC, c.CustomerName;

|CustomerName|OrderCount2016|
|------------|--------------|
|Wingtip Toys (Mayhill, NM)|30|
|Emily Whittle|28|
|Tailspin Toys (Arietta, NY)|27|
|Tailspin Toys (Good Hart, MI)|27|
|Tailspin Toys (Tierra Verde, FL)|27|
|Wingtip Toys (North Beach Haven, NJ)|27|
|Tailspin Toys (Kerby, OR)|26|
|Tailspin Toys (Vidrine, LA)|26|
|Daniel Martensson|25|
|Tailspin Toys (Marcell, MN)|25|


---

6 - Average Order Value per Customer (2016)

Proposition: For each customer, show total sales, number of orders, and average order value in 2016.

Functional Specification:

- Join Orders o → OrderLines ol → Customers c.
- SUM(ol.UnitPrice * ol.Quantity) for total sales.
- COUNT(DISTINCT o.OrderID) for number of orders.
- Compute AvgOrderValue = TotalSales / OrderCount.

In [ ]:
SELECT
    c.CustomerName,
    SUM(ol.UnitPrice * ol.Quantity) AS TotalSales2016,
    COUNT(DISTINCT o.OrderID)       AS OrderCount2016,
    CAST(SUM(ol.UnitPrice * ol.Quantity) AS decimal(18,2))
      / NULLIF(COUNT(DISTINCT o.OrderID), 0) AS AvgOrderValue2016
FROM Sales.Orders AS o
JOIN Sales.OrderLines AS ol ON ol.OrderID = o.OrderID
JOIN Sales.Customers   AS c  ON c.CustomerID = o.CustomerID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY c.CustomerName
ORDER BY TotalSales2016 DESC;


|CustomerName|TotalSales2016|OrderCount2016|AvgOrderValue2016|
|------------|--------------|--------------|-----------------|
|Tailspin Toys (Arietta, NY)|91923.70|27|3404.5814814814814|
|Tailspin Toys (Good Hart, MI)|91799.40|27|3399.9777777777777|
|Wingtip Toys (Obetz, OH)|87221.90|25|3488.8760000000000|
|Wingtip Toys (North Beach Haven, NJ)|85923.70|27|3182.3592592592592|
|Wingtip Toys (Leathersville, GA)|79706.30|22|3623.0136363636363|
|Emily Whittle|77876.55|28|2781.3053571428571|
|Tailspin Toys (Inguadona, MN)|75954.50|20|3797.7250000000000|
|Tailspin Toys (Orrtanna, PA)|74792.80|20|3739.6400000000000|
|Sabine Alksne|74440.00|24|3101.6666666666666|
|Kumar Naicker|74082.25|20|3704.1125000000000|
|Wingtip Toys (Ware Shoals, SC)|73513.60|17|4324.3294117647058|
|Wingtip Toys (Cale, AR)|73473.25|19|3867.0131578947368|
|Daniel Martensson|73304.55|25|2932.1820000000000|
|Wingtip Toys (Bourneville, OH)|72615.10|24|3025.6291666666666|
|Tailspin Toys (Kerby, OR)|71668.40|26|2756.4769230769230|
|Wingtip Toys (Rich Creek, VA)|69381.20|19|3651.6421052631578|
|Wingtip Toys (Floriston, CA)|69143.10|20|3457.1550000000000|
|Volkan senturk|68505.50|22|3113.8863636363636|
|Wingtip Toys (Indian Creek, IL)|68460.00|21|3260.0000000000000|
|Nasrin Omidzadeh|68264.15|19|3592.8500000000000|
|Yves Belisle|67842.15|20|3392.1075000000000|
|Wingtip Toys (Marion Junction, AL)|67700.15|24|2820.8395833333333|
|Raj Verma|67633.15|18|3757.3972222222222|
|Wingtip Toys (Bethel Acres, OK)|67303.90|18|3739.1055555555555|
|Tailspin Toys (Vidrine, LA)|66582.85|26|2560.8788461538461|
|Wingtip Toys (Bell Acres, PA)|66427.00|17|3907.4705882352941|
|Damodar Shenoy|65874.35|20|3293.7175000000000|
|Tailspin Toys (Sentinel Butte, ND)|65677.55|21|3127.5023809523809|
|Tailspin Toys (Furley, KS)|65330.70|20|3266.5350000000000|
|Wingtip Toys (Connoquenessing, PA)|64260.90|18|3570.0500000000000|
|Leonardo Folliero|64163.85|20|3208.1925000000000|
|Tailspin Toys (Page City, KS)|63310.05|20|3165.5025000000000|
|David Jaramillo|63242.30|24|2635.0958333333333|
|Wingtip Toys (Cuyamungue, NM)|62876.60|19|3309.2947368421052|
|Bhagavateeprasaad Malladi|61721.65|23|2683.5500000000000|
|David safranek|61502.90|19|3236.9947368421052|
|Tailspin Toys (Hodgdon, ME)|61278.40|21|2918.0190476190476|
|Arijit Bhuiyan|61191.30|14|4370.8071428571428|
|Tailspin Toys (Victory Gardens, NJ)|60758.60|17|3574.0352941176470|
|Philip Walker|60410.60|20|3020.5300000000000|
|Wingtip Toys (Jeromesville, OH)|59929.70|19|3154.1947368421052|
|Daakshaayaani Sankaramanchi|59528.70|23|2588.2043478260869|
|Tailspin Toys (Fieldbrook, CA)|59355.90|21|2826.4714285714285|
|Wingtip Toys (Broomtown, AL)|59349.20|23|2580.4000000000000|
|Wingtip Toys (Mayhill, NM)|59105.35|30|1970.1783333333333|
|Tailspin Toys (Hedrick, IA)|59086.20|19|3109.8000000000000|
|Gasper Havzija|59085.80|19|3109.7789473684210|
|Tailspin Toys (La Cueva, NM)|58348.35|19|3070.9657894736842|
|Wingtip Toys (Birds, IL)|58291.55|15|3886.1033333333333|
|Ivan Sepulveda|58135.40|18|3229.7444444444444|
|Abhra Ganguly|57951.35|19|3050.0710526315789|
|Harsha Huq|57772.25|21|2751.0595238095238|
|Wingtip Toys (Homer City, PA)|57685.40|14|4120.3857142857142|
|Wingtip Toys (Rosa Sánchez, PR)|57306.60|17|3370.9764705882352|
|Adriana Pena|56516.80|23|2457.2521739130434|
|Tailspin Toys (Topstone, CT)|56444.90|22|2565.6772727272727|
|Tailspin Toys (Muir, MI)|54969.70|20|2748.4850000000000|
|Ingrida Zeltina|54957.55|14|3925.5392857142857|
|Joel Carrillo|54874.70|15|3658.3133333333333|
|Jaroslav Fisar|54552.40|14|3896.6000000000000|
|Erik Malk|54421.40|12|4535.1166666666666|
|Tailspin Toys (Maypearl, TX)|54263.95|20|2713.1975000000000|
|Tailspin Toys (Eulaton, AL)|54225.35|20|2711.2675000000000|
|Jayanta Thakur|54034.20|17|3178.4823529411764|
|Wingtip Toys (Port Hueneme, CA)|54017.90|17|3177.5235294117647|
|Tailspin Toys (Diablock, KY)|53794.20|12|4482.8500000000000|
|Wingtip Toys (Islip Terrace, NY)|53740.90|11|4885.5363636363636|
|Tailspin Toys (Rothsville, PA)|53669.65|20|2683.4825000000000|
|Amarasimha Vinjamuri|53290.00|15|3552.6666666666666|
|Wingtip Toys (Beals, ME)|53168.95|14|3797.7821428571428|
|Daakshaayaani Kommineni|53154.00|18|2953.0000000000000|
|Tailspin Toys (Marcell, MN)|52883.80|25|2115.3520000000000|
|Tailspin Toys (King Cove, AK)|52821.20|17|3107.1294117647058|
|Tailspin Toys (North Cowden, TX)|52349.55|21|2492.8357142857142|
|Tailspin Toys (Rafael Capó, PR)|52310.15|21|2490.9595238095238|
|Veronika Necesana|52308.35|15|3487.2233333333333|
|Crina Grasu|51932.90|21|2472.9952380952380|
|Wingtip Toys (Lynne, FL)|51894.10|16|3243.3812500000000|
|Wingtip Toys (Cache, OK)|51834.95|14|3702.4964285714285|
|Tailspin Toys (Sauquoit, NY)|51821.05|9|5757.8944444444444|
|Tailspin Toys (Belgreen, AL)|51714.65|17|3042.0382352941176|
|Tailspin Toys (Cortaro, AZ)|51545.90|18|2863.6611111111111|
|Aive Petrov|51406.00|21|2447.9047619047619|
|Tailspin Toys (Arrow Rock, MO)|50967.15|23|2215.9630434782608|
|Wingtip Toys (Lilbourn, MO)|50954.20|18|2830.7888888888888|
|Tailspin Toys (Big Moose, NY)|50931.60|12|4244.3000000000000|
|Chuan Wattanasin|50488.65|17|2969.9205882352941|
|Jack Walker|50446.80|18|2802.6000000000000|
|Tailspin Toys (Fishtail, MT)|50346.55|22|2288.4795454545454|
|Dominic Davignon|50262.15|17|2956.5970588235294|
|Tailspin Toys (Wappingers Falls, NY)|50245.80|16|3140.3625000000000|
|Tailspin Toys (Optimo, NM)|50145.45|12|4178.7875000000000|
|Wingtip Toys (Keosauqua, IA)|50072.70|20|2503.6350000000000|
|Isidora Morales|50020.10|18|2778.8944444444444|
|Maryann Huddleston|49993.35|17|2940.7852941176470|
|Jai Lamble|49929.10|14|3566.3642857142857|
|Wingtip Toys (Rose Tree, PA)|49898.90|22|2268.1318181818181|
|Wingtip Toys (Omer, MI)|49860.40|17|2932.9647058823529|
|Wingtip Toys (Grabill, IN)|49845.70|17|2932.1000000000000|
|Tailspin Toys (Tierra Verde, FL)|49569.50|27|1835.9074074074074|
|Anindya Ghatak|49487.65|16|3092.9781250000000|
|Wingtip Toys (Gargatha, VA)|49474.05|19|2603.8973684210526|
|Wingtip Toys (Lytle, TX)|49416.90|21|2353.1857142857142|
|Tailspin Toys (McCamey, TX)|49143.50|12|4095.2916666666666|
|Wingtip Toys (Lucasville, OH)|48977.95|22|2226.2704545454545|
|Satish Mittal|48931.95|12|4077.6625000000000|
|Aishwarya Dantuluri|48913.95|19|2574.4184210526315|
|Dinh Mai|48898.80|17|2876.4000000000000|
|Bishwa Chatterjee|48859.80|23|2124.3391304347826|
|Tailspin Toys (Donner, LA)|48851.00|13|3757.7692307692307|
|Wingtip Toys (Silver Plume, CO)|48848.50|15|3256.5666666666666|
|Tailspin Toys (Caselton, NV)|48837.45|17|2872.7911764705882|
|Wingtip Toys (East Mountain, TX)|48768.60|14|3483.4714285714285|
|Baran Jonsson|48721.55|12|4060.1291666666666|
|Tailspin Toys (Howells, NE)|48532.55|16|3033.2843750000000|
|Tailspin Toys (Nanafalia, AL)|48512.20|22|2205.1000000000000|
|Tailspin Toys (Frankewing, TN)|48496.80|18|2694.2666666666666|
|Dena Glissen|48419.15|14|3458.5107142857142|
|Olafs Rozitis|48316.00|10|4831.6000000000000|
|Wingtip Toys (Sayner, WI)|48290.40|22|2195.0181818181818|
|Wingtip Toys (Plaquemine, LA)|48150.30|16|3009.3937500000000|
|Tailspin Toys (Heilwood, PA)|48126.35|15|3208.4233333333333|
|Urve Kasesalu|47959.70|13|3689.2076923076923|
|Wingtip Toys (Queen Valley, AZ)|47944.10|17|2820.2411764705882|
|Wingtip Toys (San Jacinto, CA)|47888.95|16|2993.0593750000000|
|Prabodh Nair|47888.40|21|2280.4000000000000|
|Neil Farrelly|47733.80|15|3182.2533333333333|
|Liidia Lepp|47639.95|17|2802.3500000000000|
|Devraj Rao|47317.30|18|2628.7388888888888|
|Ganesh Majumdar|47216.25|19|2485.0657894736842|
|Wingtip Toys (Yaak, MT)|47073.75|15|3138.2500000000000|
|Tailspin Toys (South Euclid, OH)|46991.25|19|2473.2236842105263|
|Irma Berzina|46806.00|16|2925.3750000000000|
|Jakub Lukes|46796.30|16|2924.7687500000000|
|Maksims Krastins|46712.45|10|4671.2450000000000|
|Wingtip Toys (Cos Cob, CT)|46626.80|18|2590.3777777777777|
|Wingtip Toys (Del Valle, TX)|46603.90|16|2912.7437500000000|
|Ida Celma|46424.60|17|2730.8588235294117|
|Olya Izmaylov|46320.85|19|2437.9394736842105|
|Chandana Shasthri|46172.55|17|2716.0323529411764|
|Phoung Cu|46094.35|15|3072.9566666666666|
|Tailspin Toys (Royal City, WA)|46068.55|15|3071.2366666666666|
|Hoc Tran|46029.20|12|3835.7666666666666|
|Wingtip Toys (Sunburg, MN)|45974.90|18|2554.1611111111111|
|Tailspin Toys (Eastchester, NY)|45972.05|22|2089.6386363636363|
|Wingtip Toys (Hollandsburg, IN)|45916.75|24|1913.1979166666666|
|Tailspin Toys (Hahira, GA)|45668.15|16|2854.2593750000000|
|Camille Authier|45453.25|16|2840.8281250000000|
|Wingtip Toys (Harkers Island, NC)|45389.30|14|3242.0928571428571|
|Chaayaadaevi Sonti|45195.30|10|4519.5300000000000|
|Wingtip Toys (Jamison, IA)|45126.35|19|2375.0710526315789|
|Kalyani Benjaree|44808.45|13|3446.8038461538461|
|Wingtip Toys (Trumansburg, NY)|44667.25|15|2977.8166666666666|
|Tailspin Toys (Bow Mar, CO)|44578.75|20|2228.9375000000000|
|Wingtip Toys (Beekmantown, NY)|44531.65|13|3425.5115384615384|
|Fabrice Cloutier|44467.90|17|2615.7588235294117|
|Tailspin Toys (Hayes Center, NE)|44226.50|18|2457.0277777777777|
|Wingtip Toys (Tea, SD)|44175.05|16|2760.9406250000000|
|Wingtip Toys (Akhiok, AK)|44094.95|18|2449.7194444444444|
|Wingtip Toys (Flomaton, AL)|44044.90|17|2590.8764705882352|
|Baalaamjali Devulapalli|43949.20|16|2746.8250000000000|
|Wingtip Toys (Coin, IA)|43910.45|14|3136.4607142857142|
|Sabine Zalite|43820.60|16|2738.7875000000000|
|Tailspin Toys (Long Meadow, MD)|43813.85|18|2434.1027777777777|
|Tailspin Toys (Severna Park, MD)|43775.85|22|1989.8113636363636|
|Nils Kaulins|43600.05|24|1816.6687500000000|
|Tailspin Toys (Coney Island, MO)|43583.60|24|1815.9833333333333|
|Tailspin Toys (Medicine Lodge, KS)|43582.00|15|2905.4666666666666|
|Tailspin Toys (Bethania, NC)|43407.20|19|2284.5894736842105|
|Shi Tu|43339.55|17|2549.3852941176470|
|Christian Couet|43337.55|17|2549.2676470588235|
|Tailspin Toys (New Lexington, OH)|43307.00|16|2706.6875000000000|
|Wingtip Toys (Tilleda, WI)|43247.80|20|2162.3900000000000|
|Tailspin Toys (Windsor Locks, CT)|43235.25|21|2058.8214285714285|
|Wingtip Toys (Accomac, VA)|43226.80|14|3087.6285714285714|
|Wingtip Toys (Caro, MI)|43226.35|16|2701.6468750000000|
|Shah Alizadeh|43171.00|19|2272.1578947368421|
|Isidora Urias|43155.75|20|2157.7875000000000|
|Wingtip Toys (Waycross, GA)|43108.70|21|2052.7952380952380|
|Wingtip Toys (Morrison Bluff, AR)|42957.80|15|2863.8533333333333|
|Wingtip Toys (Crossroads, NM)|42882.45|18|2382.3583333333333|
|Wingtip Toys (Cadogan, PA)|42773.45|18|2376.3027777777777|
|Aleksandrs Riekstins|42744.90|22|1942.9500000000000|
|Chompoo Atitarn|42744.35|17|2514.3735294117647|
|Tailspin Toys (Lake Erie Beach, NY)|42685.00|19|2246.5789473684210|
|Wingtip Toys (Ruthsburg, MD)|42554.55|20|2127.7275000000000|
|Wingtip Toys (Necedah, WI)|42516.60|11|3865.1454545454545|
|Ian Olofsson|42499.85|15|2833.3233333333333|
|Wingtip Toys (Mooringsport, LA)|42478.50|12|3539.8750000000000|
|Tailspin Toys (Alstead, NH)|42419.00|11|3856.2727272727272|
|Dhaatri Chavva|42380.45|19|2230.5500000000000|
|Tailspin Toys (Wimbledon, ND)|42240.05|18|2346.6694444444444|
|Tailspin Toys (Trentwood, WA)|42085.30|22|1912.9681818181818|
|Tailspin Toys (Saint Louis Park, MN)|41916.50|11|3810.5909090909090|
|Gabriela Hernandes|41725.20|14|2980.3714285714285|
|Hai Banh|41720.80|13|3209.2923076923076|
|Shantanu Huq|41650.10|20|2082.5050000000000|
|Libuse Valentova|41609.45|15|2773.9633333333333|
|Nada Ana Slosar|41608.25|20|2080.4125000000000|
|Vladimir Henzl|41568.30|14|2969.1642857142857|


---

7 - Largest Orders by Value (Top 10 in 2016)

Proposition: Find the top 10 orders by total order value in 2016.

Functional Specification:

- Sum UnitPrice * Quantity per OrderID (join Orders + OrderLines).
- Filter orders by year.
- Order by the computed total descending; take top 10.


In [ ]:
SELECT TOP 10
    o.OrderID,
    o.OrderDate,
    c.CustomerName,
    SUM(ol.UnitPrice * ol.Quantity) AS OrderTotal
FROM Sales.Orders AS o
JOIN Sales.OrderLines AS ol ON ol.OrderID = o.OrderID
JOIN Sales.Customers   AS c  ON c.CustomerID = o.CustomerID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY o.OrderID, o.OrderDate, c.CustomerName
ORDER BY OrderTotal DESC, o.OrderID;


|OrderID|OrderDate|CustomerName|OrderTotal|
|-------|---------|------------|----------|
|66991|2016-02-20|Tailspin Toys (Good Hart, MI)|29536.00|
|69920|2016-04-07|Raj Verma|27382.00|
|72045|2016-05-07|Wingtip Toys (Leathersville, GA)|27366.00|
|73521|2016-05-31|Wingtip Toys (Birds, IL)|24592.00|
|68735|2016-03-21|Tailspin Toys (Sentinel Butte, ND)|23668.80|
|64476|2016-01-07|Kalyani Benjaree|22119.60|
|64742|2016-01-12|Tailspin Toys (Arietta, NY)|21999.00|
|72296|2016-05-12|Wingtip Toys (Obetz, OH)|21554.00|
|66101|2016-02-03|Sabine Alksne|20966.00|
|71968|2016-05-06|Olafs Rozitis|20763.00|


---

8 - Orders per Salesperson (count) in 2016

Proposition: Count how many orders each salesperson handled in 2016.

Functional Specification:

- Join Sales.Orders o to Person.People p via o.SalespersonPersonID = p.PersonID.
- Filter to 2016; COUNT(*) per salesperson.

In [ ]:
SELECT
    p.FullName AS Salesperson,
    COUNT(*)   AS OrdersHandled2016
FROM Sales.Orders AS o
JOIN Application.People AS p ON p.PersonID = o.SalespersonPersonID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY p.FullName
ORDER BY OrdersHandled2016 DESC, p.FullName;

|Salesperson|OrdersHandled2016|
|-----------|-----------------|
|Taj Shand|1001|
|Archer Lamble|999|
|Hudson Hollinworth|993|
|Jack Potter|992|
|Hudson Onslow|966|
|Kayla Woodcock|958|
|Amy Trefl|941|
|Anthony Grosse|939|
|Lily Code|921|
|Sophia Hinton|907|


---

9 - January 2016 Orders with Customer & Salesperson

Proposition: List all orders in January 2016 with customer and salesperson names.

Functional Specification:

- Join Orders o to Customers c and People p.
- Date filter: OrderDate between 2016-01-01 and 2016-01-31.
- Return a clean result set.

In [ ]:
SELECT
    o.OrderID,
    o.OrderDate,
    c.CustomerName,
    p.FullName AS Salesperson
FROM Sales.Orders AS o
JOIN Sales.Customers AS c ON c.CustomerID = o.CustomerID
JOIN Application.People  AS p ON p.PersonID = o.SalespersonPersonID
WHERE o.OrderDate >= '2016-01-01'
  AND o.OrderDate <  '2016-02-01'
ORDER BY o.OrderDate, o.OrderID;


|OrderID|OrderDate|CustomerName|Salesperson|
|-------|---------|------------|-----------|
|63968|2016-01-01|Matteo Cattaneo|Hudson Onslow|
|63969|2016-01-01|Tailspin Toys (Sans Souci, SC)|Hudson Hollinworth|
|63970|2016-01-01|Wingtip Toys (Birds, IL)|Sophia Hinton|
|63971|2016-01-01|Wingtip Toys (Berville, MI)|Hudson Hollinworth|
|63972|2016-01-01|Malorie Bousquet|Anthony Grosse|
|63973|2016-01-01|Tailspin Toys (Windsor Locks, CT)|Lily Code|
|63974|2016-01-01|Leyla Asef zade|Kayla Woodcock|
|63975|2016-01-01|Tailspin Toys (Heilwood, PA)|Jack Potter|
|63976|2016-01-01|Jackson Kolios|Hudson Onslow|
|63977|2016-01-01|Mauno Laurila|Amy Trefl|
|63978|2016-01-01|Wingtip Toys (Molalla, OR)|Hudson Hollinworth|
|63979|2016-01-01|Wingtip Toys (Indian Creek, IL)|Archer Lamble|
|63980|2016-01-01|Wingtip Toys (East Fultonham, OH)|Taj Shand|
|63981|2016-01-01|Tailspin Toys (Long Meadow, MD)|Jack Potter|
|63982|2016-01-01|Wingtip Toys (Delray, WV)|Archer Lamble|
|63983|2016-01-01|Wingtip Toys (Mayhill, NM)|Hudson Onslow|
|63984|2016-01-01|Tailspin Toys (South La Paloma, TX)|Sophia Hinton|
|63985|2016-01-01|Gabriela Hernandes|Lily Code|
|63986|2016-01-01|Tailspin Toys (Hodgdon, ME)|Sophia Hinton|
|63987|2016-01-01|Suparna Bhattacharya|Kayla Woodcock|
|63988|2016-01-01|Wingtip Toys (Ruthsburg, MD)|Anthony Grosse|
|63989|2016-01-01|Tailspin Toys (Hedrick, IA)|Anthony Grosse|
|63990|2016-01-01|Sumati Chatterjee|Amy Trefl|
|63991|2016-01-01|Tailspin Toys (East Fultonham, OH)|Lily Code|
|63992|2016-01-01|Wingtip Toys (Dickworsham, TX)|Hudson Onslow|
|63993|2016-01-01|Jitka Necesana|Taj Shand|
|63994|2016-01-01|Wingtip Toys (Ovilla, TX)|Archer Lamble|
|63995|2016-01-01|Wingtip Toys (Montoya, NM)|Anthony Grosse|
|63996|2016-01-01|Tailspin Toys (Wallagrass, ME)|Archer Lamble|
|63997|2016-01-01|Wingtip Toys (Cuyamungue, NM)|Lily Code|
|63998|2016-01-01|Tailspin Toys (Valdese, NC)|Sophia Hinton|
|63999|2016-01-01|Tailspin Toys (Good Hart, MI)|Amy Trefl|
|64000|2016-01-01|Wingtip Toys (Dickerson, MD)|Jack Potter|
|64001|2016-01-01|Tailspin Toys (Fishtail, MT)|Anthony Grosse|
|64002|2016-01-01|Tailspin Toys (Eastchester, NY)|Lily Code|
|64003|2016-01-01|Tailspin Toys (East Dailey, WV)|Hudson Onslow|
|64004|2016-01-01|Nicolo Cattaneo|Jack Potter|
|64005|2016-01-01|Tailspin Toys (Koontzville, WA)|Sophia Hinton|
|64006|2016-01-01|Wingtip Toys (Glen Ullin, ND)|Archer Lamble|
|64007|2016-01-01|Wingtip Toys (Isabela, PR)|Anthony Grosse|
|64008|2016-01-01|Emilie Hrdlickova|Taj Shand|
|64010|2016-01-01|Wingtip Toys (Mayhill, NM)|Hudson Onslow|
|64011|2016-01-01|Wingtip Toys (Ovilla, TX)|Archer Lamble|
|64012|2016-01-01|Wingtip Toys (Cuyamungue, NM)|Lily Code|
|64013|2016-01-01|Wingtip Toys (Dickerson, MD)|Jack Potter|
|64014|2016-01-01|Tailspin Toys (Fishtail, MT)|Anthony Grosse|
|64015|2016-01-01|Tailspin Toys (Eastchester, NY)|Lily Code|
|64016|2016-01-02|Maksims Krastins|Hudson Onslow|
|64017|2016-01-02|Wingtip Toys (Waycross, GA)|Sophia Hinton|
|64018|2016-01-02|Tailspin Toys (Imlaystown, NJ)|Hudson Onslow|
|64019|2016-01-02|Dipti Shah|Hudson Hollinworth|
|64020|2016-01-02|Wingtip Toys (Harkers Island, NC)|Taj Shand|
|64021|2016-01-02|Tailspin Toys (Rockwall, TX)|Hudson Onslow|
|64022|2016-01-02|Matteo Cattaneo|Kayla Woodcock|
|64023|2016-01-02|Tailspin Toys (Sauquoit, NY)|Amy Trefl|
|64024|2016-01-02|Jakub Lukes|Hudson Onslow|
|64025|2016-01-02|Tailspin Toys (Hedrick, IA)|Kayla Woodcock|
|64026|2016-01-02|Pavel Bogdanov|Sophia Hinton|
|64027|2016-01-02|Shyam Poddar|Hudson Hollinworth|
|64028|2016-01-02|Francisca Laureano|Archer Lamble|
|64029|2016-01-02|Anna Gyarmathi|Anthony Grosse|
|64030|2016-01-02|Aleksandrs Riekstins|Archer Lamble|
|64031|2016-01-02|Nguyen Banh|Hudson Hollinworth|
|64032|2016-01-02|Tailspin Toys (Kalvesta, KS)|Hudson Hollinworth|
|64033|2016-01-02|Wingtip Toys (Necedah, WI)|Sophia Hinton|
|64034|2016-01-02|Kalyani Benjaree|Kayla Woodcock|
|64035|2016-01-02|Lakshmi Benipal|Sophia Hinton|
|64036|2016-01-02|Wingtip Toys (Raton, NM)|Archer Lamble|
|64037|2016-01-02|Wingtip Toys (Baldwin City, KS)|Sophia Hinton|
|64038|2016-01-02|David safranek|Jack Potter|
|64039|2016-01-02|Tailspin Toys (Maypearl, TX)|Taj Shand|
|64040|2016-01-02|Wingtip Toys (Salt Wells, NV)|Hudson Hollinworth|
|64041|2016-01-02|Urve Kasesalu|Hudson Hollinworth|
|64042|2016-01-02|Tailspin Toys (Avenal, CA)|Archer Lamble|
|64043|2016-01-02|Maksims Krastins|Lily Code|
|64044|2016-01-02|Tailspin Toys (Tooele, UT)|Kayla Woodcock|
|64045|2016-01-02|Jackson Kolios|Sophia Hinton|
|64046|2016-01-02|Tailspin Toys (Severna Park, MD)|Amy Trefl|
|64047|2016-01-02|Wingtip Toys (Glancy, MS)|Anthony Grosse|
|64048|2016-01-02|Johanna Hoornstra|Kayla Woodcock|
|64049|2016-01-02|Tailspin Toys (Boyden Arbor, SC)|Hudson Onslow|
|64050|2016-01-02|Miriam House|Sophia Hinton|
|64051|2016-01-02|David Novacek |Lily Code|
|64052|2016-01-02|Wingtip Toys (Jeromesville, OH)|Taj Shand|
|64053|2016-01-02|Gopalgobinda Sikdar|Sophia Hinton|
|64054|2016-01-02|Wingtip Toys (Cowlington, OK)|Taj Shand|
|64055|2016-01-02|Wingtip Toys (Jeromesville, OH)|Lily Code|
|64056|2016-01-02|Wingtip Toys (Necedah, WI)|Archer Lamble|
|64057|2016-01-02|Mahavir Sonkar|Lily Code|
|64058|2016-01-02|Tailspin Toys (Eden Valley, MN)|Hudson Hollinworth|
|64059|2016-01-02|Nils Kaulins|Taj Shand|
|64060|2016-01-02|Tailspin Toys (Guin, AL)|Anthony Grosse|
|64061|2016-01-02|Tailspin Toys (Aceitunas, PR)|Sophia Hinton|
|64062|2016-01-02|Francisca Laureano|Archer Lamble|
|64063|2016-01-02|Wingtip Toys (Baldwin City, KS)|Sophia Hinton|
|64064|2016-01-02|David safranek|Jack Potter|
|64065|2016-01-02|Miriam House|Sophia Hinton|
|64066|2016-01-02|Wingtip Toys (Cowlington, OK)|Taj Shand|
|64067|2016-01-02|Mahavir Sonkar|Lily Code|
|64068|2016-01-04|Wingtip Toys (North Beach Haven, NJ)|Amy Trefl|
|64069|2016-01-04|Meera Patel|Hudson Hollinworth|
|64070|2016-01-04|Tailspin Toys (Madrone, NM)|Lily Code|
|64071|2016-01-04|Tailspin Toys (El Centro, CA)|Jack Potter|
|64072|2016-01-04|Wingtip Toys (Bourbonnais, IL)|Hudson Onslow|
|64073|2016-01-04|Meera Patel|Hudson Onslow|
|64074|2016-01-04|Yves Belisle|Kayla Woodcock|
|64075|2016-01-04|Wingtip Toys (San Jacinto, CA)|Jack Potter|
|64076|2016-01-04|Dena Glissen|Archer Lamble|
|64077|2016-01-04|David Novacek |Amy Trefl|
|64078|2016-01-04|Jayanta Thakur|Archer Lamble|
|64079|2016-01-04|Tailspin Toys (Topstone, CT)|Archer Lamble|
|64080|2016-01-04|Taj Syme|Kayla Woodcock|
|64081|2016-01-04|Tailspin Toys (Tunnelhill, PA)|Jack Potter|
|64082|2016-01-04|Daakshaayaani Kommineni|Sophia Hinton|
|64083|2016-01-04|In-Su Bae|Anthony Grosse|
|64084|2016-01-04|Wingtip Toys (Balko, OK)|Archer Lamble|
|64085|2016-01-04|Matteo Cattaneo|Kayla Woodcock|
|64086|2016-01-04|Sercan Celik|Kayla Woodcock|
|64087|2016-01-04|Tailspin Toys (Tumacacori, AZ)|Lily Code|
|64088|2016-01-04|Tailspin Toys (Valdese, NC)|Jack Potter|
|64089|2016-01-04|Wingtip Toys (Mickleton, NJ)|Hudson Hollinworth|
|64090|2016-01-04|Wingtip Toys (Obetz, OH)|Kayla Woodcock|
|64091|2016-01-04|Tailspin Toys (Tierra Verde, FL)|Hudson Hollinworth|
|64092|2016-01-04|Wingtip Toys (Ware Shoals, SC)|Jack Potter|
|64093|2016-01-04|Wingtip Toys (West Frostproof, FL)|Taj Shand|
|64094|2016-01-04|Tailspin Toys (Premont, TX)|Sophia Hinton|
|64095|2016-01-04|Tailspin Toys (South Euclid, OH)|Taj Shand|
|64096|2016-01-04|Tailspin Toys (Slanesville, WV)|Amy Trefl|
|64097|2016-01-04|Dhanishta Pullela|Jack Potter|
|64098|2016-01-04|Alinne Matos|Kayla Woodcock|
|64099|2016-01-04|Jakub Lukes|Archer Lamble|
|64100|2016-01-04|Jackson Kolios|Jack Potter|
|64101|2016-01-04|Wingtip Toys (Tea, SD)|Hudson Onslow|
|64102|2016-01-04|Wingtip Toys (Isabela, PR)|Hudson Hollinworth|
|64103|2016-01-04|Bishwa Chatterjee|Archer Lamble|
|64104|2016-01-04|Mahavir Sonkar|Archer Lamble|
|64105|2016-01-04|Sointu Savonheimo|Hudson Onslow|
|64106|2016-01-04|Bhaavan Rai|Anthony Grosse|
|64107|2016-01-04|Wingtip Toys (Bozeman Hot Springs, MT)|Lily Code|
|64108|2016-01-04|Tailspin Toys (Slanesville, WV)|Hudson Onslow|
|64109|2016-01-04|Marcela Lucescu|Archer Lamble|
|64110|2016-01-04|Alvin Bollinger|Sophia Hinton|
|64111|2016-01-04|Roko Ilic|Kayla Woodcock|
|64112|2016-01-04|Eva Schulteisz|Kayla Woodcock|
|64113|2016-01-04|Tailspin Toys (Malott, WA)|Sophia Hinton|
|64114|2016-01-04|Wingtip Toys (Nichols Hills, OK)|Anthony Grosse|
|64115|2016-01-04|Tailspin Toys (Sinclair, WY)|Jack Potter|
|64116|2016-01-04|Tailspin Toys (Scofield, UT)|Hudson Hollinworth|
|64117|2016-01-04|Tailspin Toys (South Laguna, CA)|Hudson Onslow|
|64118|2016-01-04|Johanna Hoornstra|Kayla Woodcock|
|64119|2016-01-04|Wingtip Toys (Morton Grove, IL)|Kayla Woodcock|
|64120|2016-01-04|Drishti Bose|Jack Potter|
|64121|2016-01-04|Wingtip Toys (Broomtown, AL)|Archer Lamble|
|64122|2016-01-04|Tailspin Toys (King Cove, AK)|Archer Lamble|
|64123|2016-01-04|Wingtip Toys (Hollandsburg, IN)|Amy Trefl|
|64124|2016-01-04|Abel Tatarescu|Anthony Grosse|
|64125|2016-01-04|Wingtip Toys (Leathersville, GA)|Archer Lamble|
|64126|2016-01-04|Sumati Chatterjee|Kayla Woodcock|
|64127|2016-01-04|Tailspin Toys (Topstone, CT)|Anthony Grosse|
|64128|2016-01-04|Pinja Jantunen|Jack Potter|
|64129|2016-01-04|Wingtip Toys (Obion, TN)|Hudson Onslow|
|64130|2016-01-04|Tailspin Toys (Antonito, CO)|Hudson Onslow|
|64131|2016-01-04|Wingtip Toys (Lime Lake, NY)|Hudson Hollinworth|
|64132|2016-01-04|Baran Jonsson|Sophia Hinton|
|64133|2016-01-04|Wingtip Toys (Yaak, MT)|Sophia Hinton|
|64134|2016-01-04|Wingtip Toys (Portales, NM)|Amy Trefl|
|64135|2016-01-04|Tailspin Toys (Cortaro, AZ)|Hudson Onslow|
|64136|2016-01-04|Sointu Savonheimo|Kayla Woodcock|
|64137|2016-01-04|Tailspin Toys (Hedrick, IA)|Lily Code|
|64138|2016-01-04|Tailspin Toys (Teutopolis, IL)|Taj Shand|
|64139|2016-01-04|Tailspin Toys (Hambleton, WV)|Lily Code|
|64140|2016-01-04|Wingtip Toys (Cape Neddick, ME)|Amy Trefl|
|64141|2016-01-04|Wingtip Toys (Grabill, IN)|Jack Potter|
|64142|2016-01-04|Tailspin Toys (La Cueva, NM)|Hudson Hollinworth|
|64143|2016-01-04|Geza Roman|Hudson Onslow|
|64144|2016-01-04|Taj Syme|Hudson Hollinworth|
|64145|2016-01-04|Tailspin Toys (Long Meadow, MD)|Taj Shand|
|64146|2016-01-04|Om Yadav|Anthony Grosse|
|64147|2016-01-04|Wingtip Toys (Edmund, WI)|Anthony Grosse|
|64148|2016-01-04|Linh Dao|Anthony Grosse|
|64149|2016-01-04|Manca Hrastovsek|Jack Potter|
|64150|2016-01-04|Wingtip Toys (Branson West, MO)|Hudson Onslow|
|64151|2016-01-04|Tailspin Toys (Kerby, OR)|Kayla Woodcock|
|64152|2016-01-04|Wingtip Toys (Sarversville, PA)|Anthony Grosse|
|64153|2016-01-04|Wingtip Toys (Lost River, ID)|Taj Shand|
|64154|2016-01-04|Tailspin Toys (Olivette, MO)|Lily Code|
|64155|2016-01-04|Wingtip Toys (Black Lick, PA)|Hudson Hollinworth|
|64156|2016-01-04|Wingtip Toys (West Frostproof, FL)|Lily Code|
|64157|2016-01-04|Emily Whittle|Hudson Onslow|
|64158|2016-01-04|Sercan Celik|Archer Lamble|
|64159|2016-01-04|Jayanta Thakur|Archer Lamble|
|64160|2016-01-04|Matteo Cattaneo|Kayla Woodcock|
|64161|2016-01-04|Jackson Kolios|Jack Potter|
|64162|2016-01-04|Tailspin Toys (Sinclair, WY)|Jack Potter|
|64163|2016-01-04|Wingtip Toys (Leathersville, GA)|Archer Lamble|
|64164|2016-01-04|Tailspin Toys (Cortaro, AZ)|Hudson Onslow|
|64249|2016-01-04|Wingtip Toys (Leathersville, GA)|Archer Lamble|
|64165|2016-01-05|Wingtip Toys (North Beach Haven, NJ)|Jack Potter|
|64166|2016-01-05|Wingtip Toys (Nuangola, PA)|Hudson Onslow|
|64167|2016-01-05|Wingtip Toys (Wapiti, WY)|Lily Code|


---

10 - Distinct Items Purchased per Customer (2016)

Proposition: For each customer, count how many different stock items they purchased in 2016; list the top 10.

Functional Specification:

- Join Orders o → OrderLines ol → Customers c.
- Filter to 2016; COUNT(DISTINCT ol.StockItemID) per customer.
- Sort by that count; take top 10.

In [ ]:
SELECT TOP 10
    c.CustomerName,
    COUNT(DISTINCT ol.StockItemID) AS DistinctItemsPurchased2016
FROM Sales.Orders AS o
JOIN Sales.OrderLines AS ol ON ol.OrderID = o.OrderID
JOIN Sales.Customers   AS c  ON c.CustomerID = o.CustomerID
WHERE YEAR(o.OrderDate) = 2016
GROUP BY c.CustomerName
ORDER BY DistinctItemsPurchased2016 DESC, c.CustomerName;


|CustomerName|DistinctItemsPurchased2016|
|------------|--------------------------|
|Tailspin Toys (Arietta, NY)|79|
|Wingtip Toys (Mayhill, NM)|75|
|Emily Whittle|73|
|Tailspin Toys (Kerby, OR)|72|
|Tailspin Toys (Fieldbrook, CA)|70|
|Tailspin Toys (Vidrine, LA)|70|
|Wingtip Toys (Broomtown, AL)|69|
|Wingtip Toys (North Beach Haven, NJ)|69|
|Bishwa Chatterjee|67|
|Tailspin Toys (Marcell, MN)|67|
